# 07 · Desplegar en Databricks Apps

Identifica solo al usuario que ejecuta el notebook y despliega en SU carpeta (`/Workspace/Users/<correo>/<app>`), nunca en la raíz del workspace.

1. `apps.get`: valida si la app existe; si no, la crea; si existe, actualiza sus recursos.
2. Recurso secreto de la App con permiso **READ** sobre el scope y la clave del token de Foundry, con el nombre `foundry-token` que usa `app.yaml` (`valueFrom`).
3. `deploy_and_wait` del código subido.

Antes: `app.yaml` con los datos de Foundry (endpoint, api-version, deployments). El notebook vive en `comparador-dbx/notebooks/`.

In [ ]:
%pip install -q databricks-sdk==0.154.0
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("app", "comparador-normativo", "Nombre de la app")
dbutils.widgets.text("secreto_scope", "", "Scope del secreto (token de Foundry)")
dbutils.widgets.text("secreto_clave", "", "Clave del secreto")
dbutils.widgets.text("ruta", "", "Carpeta del workspace (vacío = la de tu usuario)")

In [ ]:
import io
import posixpath
from pathlib import Path

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.workspace import ImportFormat
from databricks.sdk.service.apps import (App, ComputeState, AppDeployment, AppDeploymentMode, AppResource, AppResourceSecret,
                                         AppResourceSecretSecretPermission)

app, scope, clave, ruta = (dbutils.widgets.get(n).strip() for n in ("app", "secreto_scope", "secreto_clave", "ruta"))
assert app and scope and clave, "Completa app, secreto_scope y secreto_clave"

origen = Path.cwd().parent                                  # comparador-dbx/
EXCLUIR_RAIZ = {"output", "documentos", "notebooks", "tests", ".git"}       # solo en la raíz: backend/output/ es código
EXCLUIR_EN_CUALQUIER_NIVEL = {"node_modules", "__pycache__", ".pytest_cache"}
EXCLUIR_ARCHIVOS = {"ejemplo.env"}                          # secretos: nunca se suben (además, todo .env* y *.pem/*.key)

w = WorkspaceClient()                                       # credenciales del propio notebook
usuario = w.current_user.me().user_name                     # el usuario que ejecuta
ruta = posixpath.normpath(ruta or f"/Workspace/Users/{usuario}/{app}")      # normpath: no admite ".." para salir de tu carpeta
assert ruta.startswith(f"/Workspace/Users/{usuario}/"), f"La ruta debe estar dentro de /Workspace/Users/{usuario}/"
print(f"Usuario: {usuario} · App: {app} · Código en: {ruta}")

In [ ]:
n = 0
for p in sorted(origen.rglob("*")):
    rel = p.relative_to(origen)
    if not p.is_file() or rel.parts[0] in EXCLUIR_RAIZ or set(rel.parts[:-1]) & EXCLUIR_EN_CUALQUIER_NIVEL or p.name in EXCLUIR_ARCHIVOS or p.name.startswith(".env") or p.name.endswith((".pyc", ".map", ".pem", ".key")):
        continue
    destino = f"{ruta}/{rel.as_posix()}"
    w.workspace.mkdirs(destino.rsplit("/", 1)[0])
    contenido = p.read_bytes() or b"\n"                      # el workspace rechaza archivos vacíos (los __init__.py)
    w.workspace.upload(destino, io.BytesIO(contenido), format=ImportFormat.RAW, overwrite=True)   # RAW: archivos tal cual, nunca notebooks
    n += 1
print(f"{n} archivos subidos")

In [ ]:
secreto = AppResource(name="foundry-token",                  # el valueFrom de app.yaml
                     secret=AppResourceSecret(scope=scope, key=clave, permission=AppResourceSecretSecretPermission.READ))
definicion = App(name=app, description="Comparador normativo de doble vía", resources=[secreto])
try:
    w.apps.get(app)                                         # ¿existe?
    w.apps.update(app, app=definicion)                      # sí: se asegura el recurso del secreto (reemplaza los recursos que se hayan agregado a mano en la consola)
    print(f"La app {app} ya existía: recurso del secreto actualizado")
except NotFound:
    w.apps.create_and_wait(definicion)
    print(f"App {app} creada")

if w.apps.get(app).compute_status.state != ComputeState.ACTIVE:      # una app detenida (la Free las apaga) no admite despliegues
    print("La app está detenida: se inicia antes de desplegar")
    w.apps.start_and_wait(app)
w.apps.deploy_and_wait(app, app_deployment=AppDeployment(source_code_path=ruta, mode=AppDeploymentMode.SNAPSHOT))
print("Listo:", w.apps.get(app).url)